In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Indiana University Chest X-rays — Data Preparation

This notebook prepares the Indiana University Chest X-ray dataset for a medical Vision-Language Model (VLM).

Workflow:
1. Brief dataset overview & EDA
2. Data Cleaning & Image-Report Mapping
3. Image Preprocessing & Quality Checks
4. Online Data Augmentation


## Import needed libraries

In [ ]:

import pandas as pd
import numpy as np
import random

from collections import Counter
import re

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import missingno as msno
import textwrap

from pathlib import Path
from IPython.display import display
from PIL import Image
import PIL.Image
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split

import os
import kagglehub

## Dataset Overview


In [ ]:
dataset_path = kagglehub.dataset_download(
    "raddar/chest-xrays-indiana-university"
)
print(dataset_path)

In [ ]:
# Locate the dataset in Kaggle
INPUT_DIR = Path("/kaggle/input/datasets/raddar/chest-xrays-indiana-university")

if not INPUT_DIR.exists():
    INPUT_DIR = Path("/kaggle/input")

report_files = list(INPUT_DIR.rglob("indiana_reports.csv"))
projection_files = list(INPUT_DIR.rglob("indiana_projections.csv"))

if not report_files or not projection_files:
    raise FileNotFoundError(
        "Could not find indiana_reports.csv and indiana_projections.csv. "
        "Make sure the Indiana University Chest X-rays dataset is attached."
    )

REPORTS_PATH = report_files[0]
PROJECTIONS_PATH = projection_files[0]

# Prefer the normalized image folder when available
image_dirs = [p for p in INPUT_DIR.rglob("images_normalized") if p.is_dir()]
if not image_dirs:
    image_dirs = [p for p in INPUT_DIR.rglob("images") if p.is_dir()]

if not image_dirs:
    raise FileNotFoundError("Could not find the dataset image directory.")

IMAGES_DIR = image_dirs[0]

# Load metadata
reports = pd.read_csv(REPORTS_PATH)
projections = pd.read_csv(PROJECTIONS_PATH)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
image_paths = sorted(
    path for path in IMAGES_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"Medical reports: {len(reports):,}")
print(f"Projection records: {len(projections):,}")
print(f"Image files found: {len(image_paths):,}")

## EDA & Cleaning

In [ ]:
reports.head()

In [ ]:
reports.info()

In [ ]:
reports.isnull().sum()

In [ ]:
reports.duplicated().sum()

In [ ]:
projections.head()

In [ ]:

projections.info()

In [ ]:
projections.duplicated().sum()

### Merging Projections and Reports

In [ ]:
df = pd.merge(reports, projections, on='uid', how='inner')
df.head()

In [ ]:
df.info()

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage': (df.isnull().sum() / len(df)) * 100
})

print(missing)

In [ ]:
filenames_no_findings = df.loc[
    df["findings"].fillna("").str.strip().eq(""), "filename"
]

print(f"Number of files with no findings: {len(filenames_no_findings)}")
display(filenames_no_findings)

In [ ]:
# Heatmap showing correlation of missingness
msno.heatmap(df)
plt.show()

In [ ]:
# Fill missing text values without rewriting existing medical content
df = df.fillna("Not available")

In [ ]:
df.isnull().sum()

### Actual image files with file names

In [ ]:
# Identify actual image files using the paths collected in the overview
actual_filenames = {path.name for path in image_paths}
print(f"Number of image files found: {len(actual_filenames)}")

### Matching Projection Metadata

In [ ]:
# Keep only images that have matching projection metadata
valid_projections = df[
    df["filename"].isin(actual_filenames)
]
print(f"Records with matching projection metadata: {len(valid_projections)}")

In [ ]:
# Exclude images without metadata
excluded_images = sorted(
    actual_filenames - set(df["filename"].dropna())
)
print(f"Images excluded (no matching metadata): {len(excluded_images)}")

In [ ]:
# Create a DataFrame of excluded records
excluded_records = pd.DataFrame({
    "filename": excluded_images,
    "reason": "No matching projection metadata"
})

excluded_records

In [ ]:
valid_projections.info()

### Samples

In [ ]:

# Create a lookup dictionary: filename -> full path
image_lookup = {path.name: path for path in image_paths}

for i in range(min(6, len(valid_projections))):
    sample = valid_projections.iloc[i]
    filename = sample["filename"]
    # Find corresponding image
    img_path = image_lookup.get(filename)
    img = PIL.Image.open(img_path)
    width, height = img.size
    # Build the raw text content
    raw_text = f"Sample {i}\n" + "="*30 + "\n\n"
    for col, value in sample.items():
        raw_text += f"{col} : {value}\n\n"
    raw_text += f"Image path: {img_path}\n"
    raw_text += f"Image dimensions: {width} x {height}"
    # Wrap long lines
    wrapped_lines = []
    for line in raw_text.split('\n'):
        if len(line) > 60:
            wrapped_lines.extend(textwrap.wrap(line, width=60))
        else:
            wrapped_lines.append(line)
    text_content = '\n'.join(wrapped_lines)
    # Create a wider figure with custom width ratios between text and image
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1.3, 1]})
    # Left subplot: Display wrapped text metadata
    axes[0].axis("off")
    axes[0].text(
        0.0, 1.0,
        text_content,
        fontsize=9,
        verticalalignment='top',
        family='monospace'
    )
    # Right subplot: Display image
    axes[1].imshow(img, cmap='gray')
    axes[1].axis("off")
    axes[1].set_title(filename)
    # Adjust spacing
    fig.subplots_adjust(left=0.05, right=0.95, top=0.92, bottom=0.05, wspace=0.1)
    plt.show()

In [ ]:

valid_projections['projection'].value_counts(normalize=True)

In [ ]:
# See which projections exist for each UID
uid_views = (
    valid_projections
    .groupby("uid")["projection"]
    .agg(lambda x: set(x))
)

# UIDs with only Frontal images
frontal_only_uids = uid_views[uid_views == {"Frontal"}].index

# UIDs with only Lateral images
lateral_only_uids = uid_views[uid_views == {"Lateral"}].index

print(f"Frontal-only UIDs: {len(frontal_only_uids)}")
print(f"Lateral-only UIDs: {len(lateral_only_uids)}")

In [ ]:
valid_projections['Problems'].unique()

### Text Length frequencies

In [ ]:
exclude_cols = ["uid", "projection", "filename"]
cols_to_plot = [col for col in valid_projections.columns if col not in exclude_cols][:7]

# Create a subplot grid
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
  lengths = valid_projections[col].astype(str).str.len()

  # Plot on the specific subplot axis
  sns.histplot(lengths, bins=30, kde=True, ax=axes[i])
  axes[i].set_title(f"Distribution of Text Length in '{col}'", fontsize=12)
  axes[i].set_xlabel("Text Length", fontsize=10)
  axes[i].set_ylabel("Count", fontsize=10)
  axes[i].grid(axis="y", linestyle="--", alpha=0.7)

# Hide any unused axes
for j in range(len(cols_to_plot), len(axes)):
  fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
problem_counts = valid_projections['Problems'].value_counts().head(15)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x=problem_counts.values,
    y=problem_counts.index,
    hue=problem_counts.index,
    palette='coolwarm',
    legend=False
)
plt.xlabel('Count of Problem', fontsize=12)
plt.ylabel('Problem report', fontsize=12)
plt.title('Top 15 Problems Distribution', fontsize=14, fontweight='bold')
plt.yticks(rotation=0)
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(
            str(int(width)),
            (width, p.get_y() + p.get_height() / 2.),
            xytext=(5, 0),
            textcoords="offset points",
            ha='left',
            va='center'
        )

plt.tight_layout()
plt.show()

## Image Preprocessing & Quality Checks

In this step, we inspect the X-ray images to ensure they are valid and suitable for further processing.

* Check image readability and file integrity.
* Identify corrupted or unsupported images.
* Analyze image dimensions and color modes.

**Note:** Original images will remain unchanged. Resizing and normalization will be performed later according to the selected model architecture.


In [ ]:
image_quality_results = []

for filename in valid_projections["filename"].drop_duplicates():
    image_path = image_lookup.get(filename)

    if image_path is None:
        image_quality_results.append({
            "filename": filename,
            "status": "Missing",
            "width": None,
            "height": None,
            "mode": None
        })
        continue

    try:
        with Image.open(image_path) as img:
            img.verify()

        with Image.open(image_path) as img:
            width, height = img.size
            mode = img.mode

        image_quality_results.append({
            "filename": filename,
            "status": "Valid",
            "width": width,
            "height": height,
            "mode": mode
        })

    except Exception:
        image_quality_results.append({
            "filename": filename,
            "status": "Corrupted",
            "width": None,
            "height": None,
            "mode": None
        })

image_quality_df = pd.DataFrame(image_quality_results)

print("Total unique mapped images:", len(image_quality_df))
print("\nImage status:")
print(image_quality_df["status"].value_counts())

print("\nImage modes:")
print(image_quality_df["mode"].value_counts(dropna=False))

print("\nImage dimensions summary:")
display(image_quality_df[["width", "height"]].describe())

print("\nSample results:")
display(image_quality_df.head())


### Image Augmentation



In [ ]:
def enhance_medical_image(img):
    # 1. Apply CLAHE for local contrast
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    st1 = clahe.apply(img)

    # 2. Apply gentle sharpening
    gaussian = cv2.GaussianBlur(st1, (0, 0), sigmaX=1.5)
    sharpened = cv2.addWeighted(st1, 1.3, gaussian, -0.3, 0)

    return sharpened

In [ ]:
random.seed(42)
image_path = random.choice(image_paths)

# Load image in grayscale (matching your dataset's 'L' mode)
original_img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# Process the image
enhanced_img = enhance_medical_image(original_img)

# Plot Before and After side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Before Plot
axes[0].imshow(original_img, cmap='gray')
axes[0].set_title('Before (Original Grayscale)', fontsize=14, fontweight='bold')
axes[0].axis('off')

# After Plot
axes[1].imshow(enhanced_img, cmap='gray')
axes[1].set_title(
    'After (CLAHE + Edge Sharpening)', fontsize=14, fontweight='bold'
)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Data Filtering

In [ ]:
df_new = valid_projections[(
    valid_projections['findings'].str.strip() != "") & (valid_projections['impression'].str.strip() != ""
    )]

In [ ]:
df_new['report_combined'] = df_new['findings'] + " " + df_new['impression']

# Calculate length of the new concatenated report
df_new['report_length'] = df_new['report_combined'].apply(len)

# Display some sample entries to verify
print("\nSample combined reports and their lengths:\n")
print(df_new[['report_combined', 'report_length']].head(10))

In [ ]:
sample_report_combined = df_new['report_combined'].sample(40, random_state=42)
print("\nSample 'report_combined' Entries:\n")
for idx, combine in enumerate(sample_report_combined):
    print(f"{idx + 1}: {combine}\n")

In [ ]:
# Plotting the distribution of report lengths
plt.figure(figsize=(14, 7))
sns.histplot(df_new['report_length'], bins=50, kde=True, color='dodgerblue')
plt.title("Distribution of Concatenated Report Lengths")
plt.xlabel("Report Length (Number of Characters)")
plt.ylabel("Frequency")
plt.axvline(df_new['report_length'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['report_length'].mean():.2f}")
plt.axvline(df_new['report_length'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['report_length'].median():.2f}")
plt.legend()
plt.tight_layout()
plt.show()

# Display summary statistics
print("\nSummary statistics for 'report_length':\n")
print(df_new['report_length'].describe())

In [ ]:
q90 = df_new['report_length'].quantile(0.90)
q95 = df_new['report_length'].quantile(0.95)
q99 = df_new['report_length'].quantile(0.99)

print("\nLength Threshold Suggestions:")
print(f"90th Percentile: {q90} characters")
print(f"95th Percentile: {q95} characters")
print(f"99th Percentile: {q99} characters")
print(f"Mean Length: {df_new['report_length'].mean():.2f} characters")
print(f"Median Length: {df_new['report_length'].median():.2f} characters")

In [ ]:
df_new['word_count'] = df_new['report_combined'].apply(lambda x: len(x.split()))

# Display a few samples to verify
print("\nSample word counts:\n")
print(df_new[['report_combined', 'word_count']].head(10))

In [ ]:
plt.figure(figsize=(14, 7))
sns.histplot(df_new['word_count'], bins=50, kde=True, color='coral')
plt.title("Distribution of Word Count in Combined Reports")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.axvline(df_new['word_count'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['word_count'].mean():.2f}")
plt.axvline(df_new['word_count'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['word_count'].median():.2f}")
plt.legend()
plt.tight_layout()
plt.show()

# Display summary statistics
print("\nSummary statistics for 'word_count':\n")
print(df_new['word_count'].describe())

In [ ]:
print(f"90th Percentile Length Threshold: {q90} characters")

# Initial number of rows before filtering
initial_rows = len(df_new)

# Apply filtering
df_new = df_new[df_new['report_length'] <= q90]

# New number of rows after filtering
filtered_rows = len(df_new)

print(f"Number of rows after filtering: {filtered_rows}")
print(f"Number of rows removed: {initial_rows - filtered_rows}")

In [ ]:
plt.figure(figsize=(14, 7))
sns.histplot(df_new['report_length'], bins=50, kde=True, color='dodgerblue')
plt.title("Distribution of Report Lengths After Removing 90th Percentile")
plt.xlabel("Report Length (Number of Characters)")
plt.ylabel("Frequency")
plt.axvline(df_new['report_length'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['report_length'].mean():.2f}")
plt.axvline(df_new['report_length'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['report_length'].median():.2f}")
plt.legend()
plt.tight_layout()

plt.figure(figsize=(14, 7))
sns.histplot(df_new['word_count'], bins=50, kde=True, color='coral')
plt.title("Distribution of Word Count After Removing 90th Percentile")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.axvline(df_new['word_count'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_new['word_count'].mean():.2f}")
plt.axvline(df_new['word_count'].median(), color='green', linestyle='--',
            label=f"Median: {df_new['word_count'].median():.2f}")
plt.legend()
plt.tight_layout()
plt.show()

# Display summary statistics
print("\nSummary statistics for 'word_count':\n")
print(df_new['word_count'].describe())

In [ ]:
sample_report_combined = df_new['report_combined'].sample(20, random_state=42)
print("\nSample 'report_combined' Entries:\n")
for idx, combine in enumerate(sample_report_combined):
    print(f"{idx + 1}: {combine}\n")

## Preprocess "report_combines"

In [ ]:
def preprocess_report_v2(report):
    # Convert to lowercase
    report = report.lower()
    # Remove non-alphabetic characters and keep only spaces
    report = re.sub(r'[^a-z\s]', ' ', report)
    report = re.sub(r'\bx{3,}\b', '', report)
    # Replace multiple spaces with a single space
    report = re.sub(r'\s+', ' ', report).strip()
    return report
# Apply preprocessing
df_new['processed_report'] = df_new['report_combined'].apply(preprocess_report_v2)

# Display some samples to verify
print("\n Sample Preprocessed Reports:\n")
print(df_new[['report_combined', 'processed_report']].head(5))

In [ ]:
sample_processed_report = df_new['processed_report'].sample(20, random_state=69)
print("\nSample 'processed_report' Entries:\n")
for idx, combine in enumerate(sample_processed_report):
    print(f"{idx + 1}: {combine}\n")

In [ ]:
all_words = " ".join(df_new['processed_report']).split()

# Count unique words and tags
unique_words = set(all_words)
word_count = len(unique_words)
print(f"Total unique words: {word_count}")

# Frequency distribution of words
word_freq = Counter(all_words)

# Top 10 most common words
print("\nTop 30 most common words:")
print(word_freq.most_common(30))

### Re-exmaining mesh

In [ ]:
# Check a few examples to confirm delimiter patterns
print("\nSample MeSH values:\n", df_new['MeSH'].head(5))

# Split by comma, semicolon, or other delimiters
mesh_terms = df_new['MeSH'].apply(lambda x: re.split(r'[;,/]', x.strip().lower()))

# Flatten the list of lists and count frequencies
all_terms = [term.strip() for sublist in mesh_terms for term in sublist if term.strip()]
term_counts = Counter(all_terms)

# Display the 15 most common terms
common_terms = term_counts.most_common(15)
print("\nTop 15 Most Common MeSH Terms:\n", common_terms)

# Plot the term frequency
terms, counts = zip(*common_terms)
plt.figure(figsize=(14, 7))
sns.barplot(x=list(counts), y=list(terms), palette="coolwarm")
plt.title("Top 15 Most Frequent MeSH Terms")
plt.xlabel("Frequency")
plt.ylabel("MeSH Terms")
plt.tight_layout()
plt.show()

### Distribution Balance

In [ ]:
normal_keywords = ['normal', 'unremarkable', 'clear', 'stable', 'within normal limits']
non_normal_keywords = ['disease', 'effusion', 'nodule', 'infiltrate', 'pneumothorax',
                       'opacity', 'abnormality', 'enlargement', 'calcification',
                       'atelectasis', 'consolidation', 'mass', 'lesion']

def classify_mesh(mesh_terms):
    """
    Classifies the MeSH column into 'normal' or 'non-normal'.
    """
    # Ensure it's a string for processing
    mesh_terms = str(mesh_terms).lower()

    # Check for normal keywords
    if any(keyword in mesh_terms for keyword in normal_keywords):
        return 'normal'

    # Check for non-normal keywords
    elif any(keyword in mesh_terms for keyword in non_normal_keywords):
        return 'non-normal'

    # Default to non-normal if no clear match
    return 'non-normal'

# Apply classification
df_new['class'] = df_new['MeSH'].apply(classify_mesh)

# Display class distribution
print("\nClass Distribution:\n", df_new['class'].value_counts())

In [ ]:
# Plotting as a Pie Chart to show dristribution
class_counts = df_new['class'].value_counts()
plt.figure(figsize=(6, 6))
colors = sns.color_palette('Set2')[0:2]

plt.pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',  # Automatically calculates and displays percentages
    startangle=140,
    colors=colors,
    textprops={'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},  # Adds a clean border
)

plt.title(
    'Distribution: Normal vs. Other Problems', fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
df_new.info()

In [ ]:
df_new.head()

In [ ]:
df_new['comparison'].unique()

In [ ]:
df_new['comparison'].value_counts()

In [ ]:
df_new['Problems'].unique()

In [ ]:
df_new['MeSH'].unique()

In [ ]:
df_new['indication'].unique()

In [ ]:
df_new = df_new.drop(columns=['comparison','image','report_combined', 'report_length',	'word_count', 'findings', 'impression'])

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return text
    text = str(text).lower()
    # Remove anonymization tokens like XXX, XXXX, XXXXX
    text = re.sub(r'\bx{3,}\b', '', text)
    # Remove punctuation
    text = re.sub(r'[^\w\s;/]', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
df_new['indication'] = df_new['indication'].apply(clean_text)

In [ ]:
df_new['Problems'] = df_new['Problems'].apply(clean_text)

In [ ]:
df_new['MeSH'] = df_new['MeSH'].apply(clean_text)

In [ ]:
df_new.head()

### Report

In [ ]:
def image_report(df_new, num_samples):
    """
    Visualize up to 1 or 2 images per report alongside the report text
    using the pre-declared image_lookup dictionary.
    """
    samples = df_new.sample(num_samples, random_state=42)

    for _, row in samples.iterrows():
        uid = row['uid']
        report = row['processed_report']

        # Get up to 2 image entries for this UID from the dataframe
        image_entries = df_new[df_new['uid'] == uid]['filename'].tolist()[:2]

        # Map directly using the pre-existing global image_lookup
        img_paths = [image_lookup[Path(img_entry).name] for img_entry in image_entries]

        num_images = len(img_paths)
        if num_images == 0:
            print(f"No images found for UID: {uid}")
            continue

        fig, axes = plt.subplots(1, num_images, figsize=(6 * num_images, 6))
        fig.suptitle(f"UID: {uid}\nReport: {report}", fontsize=10, ha='center', wrap=True)

        if num_images == 1:
            axes = [axes]

        for idx, img_path in enumerate(img_paths):
            original_img = cv2.imread(str(img_path),cv2.IMREAD_GRAYSCALE)
            image = enhance_medical_image(original_img)

            axes[idx].imshow(image, cmap='gray')
            axes[idx].axis('off')
            axes[idx].set_title(f"Image {idx + 1}")

        plt.tight_layout()
        plt.show()

In [ ]:
image_report(df_new, num_samples=2)

## Splitting

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd

drive.mount("/content/drive")

SPLIT_DIR = Path("/content/drive/MyDrive/cxr_shared")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = SPLIT_DIR / "split_manifest.csv"

if SPLIT_PATH.exists():
    split_manifest = pd.read_csv(SPLIT_PATH)
    split_map = dict(zip(split_manifest["uid"], split_manifest["split"]))

    new_uids = set(df_new["uid"].dropna().unique()) - set(split_map)
    if new_uids:
        raise ValueError(
            f"{len(new_uids)} studies aren't in the saved split. "
            "Make sure both notebooks use the same cleaned dataset."
        )
else:
    uids = np.array(sorted(df_new["uid"].dropna().unique()))
    rng = np.random.default_rng(42)
    rng.shuffle(uids)

    train_end = int(0.80 * len(uids))
    val_end = int(0.90 * len(uids))

    split_manifest = pd.DataFrame({
        "uid": uids,
        "split": (
            ["train"] * train_end
            + ["validation"] * (val_end - train_end)
            + ["test"] * (len(uids) - val_end)
        ),
    })
    split_manifest.to_csv(SPLIT_PATH, index=False)
    split_map = dict(zip(split_manifest["uid"], split_manifest["split"]))

df_new = df_new.copy()
df_new["split"] = df_new["uid"].map(split_map)

train_df = df_new[df_new["split"] == "train"].copy().reset_index(drop=True)
val_df = df_new[df_new["split"] == "validation"].copy().reset_index(drop=True)
test_df = df_new[df_new["split"] == "test"].copy().reset_index(drop=True)

assert set(train_df.uid).isdisjoint(val_df.uid)
assert set(train_df.uid).isdisjoint(test_df.uid)
assert set(val_df.uid).isdisjoint(test_df.uid)

print("Split manifest:", SPLIT_PATH)
print("Studies:", split_manifest["split"].value_counts().to_dict())
print("Rows:", len(train_df), len(val_df), len(test_df))

In [ ]:
print("\nClass Distribution in Training Set:\n", train_df['class'].value_counts())
print("\nClass Distribution in Validation Set:\n", val_df['class'].value_counts())
print("\nClass Distribution in Testing Set:\n", test_df['class'].value_counts())

In [ ]:
train_df.iloc[3]

### Save splits (so a disconnect doesn't force you to redo data prep)

In [ ]:
from pathlib import Path
SAVE_DIR = Path("/content/medgemma_cxr")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/medgemma_cxr")
except Exception as e:
    print("Drive not mounted, using local disk:", e)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("SAVE_DIR =", SAVE_DIR)

In [ ]:
train_df.to_csv(SAVE_DIR / "train.csv", index=False)
val_df.to_csv(SAVE_DIR / "val.csv", index=False)
test_df.to_csv(SAVE_DIR / "test.csv", index=False)
print("Saved splits to", SAVE_DIR)

# Model — MedGemma 4B: Zero-Shot vs. LoRA Fine-Tuned

**Every cell is small and resumable.** After a disconnect: run sections 1–2, then jump to the section you were in.
Progress (baseline eval, LoRA checkpoints, fine-tuned eval) is saved to `SAVE_DIR` (Google Drive if mounted).

Research/demo use only — not for clinical decisions.

## 1. Install & login

In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate peft rouge-score nltk "bitsandbytes>=0.46.1"

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # token with access to google/medgemma-4b-it

## 2. Setup / reload state — run this after EVERY reconnect

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/medgemma_cxr")
SAVE_DIR = Path("/content/drive/MyDrive/qwen3vl_cxr")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
PROMPT_TEXT = "Describe the findings of this chest X-ray."
print("Data:", DATA_DIR)
print("Qwen results:", SAVE_DIR)

In [ ]:
import pandas as pd
from pathlib import Path

if "train_df" not in globals():
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    val_df = pd.read_csv(DATA_DIR / "val.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

if "image_lookup" not in globals():
    import kagglehub

    ds = Path(kagglehub.dataset_download(
        "raddar/chest-xrays-indiana-university"
    ))
    image_lookup = {
        p.name: p
        for p in ds.rglob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    }

print(len(train_df), len(val_df), len(test_df), "| images:", len(image_lookup))

## 3. Helpers (generate + evaluate, evaluation resumes from its CSV)

In [ ]:
import torch
from PIL import Image

MAX_NEW = 160

def load_img(fname):
    return Image.open(image_lookup[fname]).convert("RGB")

def generate_report(model, processor, image, max_new_tokens=MAX_NEW):
    msgs = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT_TEXT},
        ],
    }]

    inputs = processor.apply_chat_template(
        msgs,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    # Qwen's model guide removes this field when the processor provides it.
    inputs.pop("token_type_ids", None)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(COMPUTE_DTYPE)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    return processor.decode(out[0, input_len:], skip_special_tokens=True).strip()

In [ ]:
def evaluate_model(model, processor, df_eval, label):
    """Scores each sample, appends to SAVE_DIR/eval_<label>.csv after every sample. Re-run to resume."""
    path = SAVE_DIR / f"eval_{label}.csv"
    rows = pd.read_csv(path).to_dict("records") if path.exists() else []
    done = {r["filename"] for r in rows}
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    smooth = SmoothingFunction().method1
    model.eval()
    for _, r in tqdm(df_eval.iterrows(), total=len(df_eval), desc=label):
        if r["filename"] in done:
            continue
        gen = generate_report(model, processor, load_img(r["filename"]))
        g, ref = norm_text(gen), r["processed_report"]
        rows.append({"filename": r["filename"],
                     "rougeL": scorer.score(ref, g)["rougeL"].fmeasure,
                     "bleu": sentence_bleu([ref.split()], g.split(), smoothing_function=smooth),
                     "generated": gen})
        pd.DataFrame(rows).to_csv(path, index=False)
        torch.cuda.empty_cache()
    res = pd.DataFrame(rows)
    m = {"rougeL": float(res.rougeL.mean()), "bleu": float(res.bleu.mean()), "n": len(res)}
    json.dump(m, open(SAVE_DIR / f"metrics_{label}.json", "w"))
    print(f"{label}: ROUGE-L={m['rougeL']:.3f}  BLEU={m['bleu']:.3f}  (n={m['n']})")
    return m

In [ ]:
NUM_EVAL_SAMPLES = 24   # frontal views only, fixed subset
eval_df = (test_df[test_df["projection"] == "Frontal"]
           .sample(NUM_EVAL_SAMPLES, random_state=42).reset_index(drop=True))
print(len(eval_df))

## 4. Baseline: zero-shot (skip if `metrics_zero_shot.json` already exists)

In [ ]:
print("baseline already done:", (SAVE_DIR / "metrics_zero_shot.json").exists())

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
COMPUTE_DTYPE = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)
model.eval()
print("Qwen loaded.")

In [ ]:
row = eval_df.iloc[0]
print("Generated:\n", generate_report(model, processor, load_img(row["filename"])))
print("\nGround truth:\n", row["processed_report"])

In [ ]:
baseline_metrics = evaluate_model(model, processor, eval_df, "zero_shot")

In [ ]:
# free the bf16 model before loading the 4-bit one
del model; gc.collect(); torch.cuda.empty_cache()
print("freed, GPU mem MB:", torch.cuda.memory_allocated() // 2**20)

## 5. Load 4-bit model + LoRA (auto-resumes from the last checkpoint)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
if "processor" not in globals():
    processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", dtype=torch.bfloat16)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
print("4-bit model loaded")

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

CKPT = SAVE_DIR / "lora_ckpt"
if (CKPT / "adapter_config.json").exists():
    model = PeftModel.from_pretrained(model, str(CKPT), is_trainable=True)
    print("Resumed LoRA from", CKPT)
else:
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                     # language-model attention only (regex), not the vision tower
                     target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj)")
    model = get_peft_model(model, cfg)
    print("Fresh LoRA")
model.print_trainable_parameters()

def lora_b_norm(m):
    return sum(p.float().norm().item() for n, p in m.named_parameters() if "lora_B" in n)
print("LoRA B norm:", lora_b_norm(model))   # 0.0 on a fresh adapter

## 6. Training data builder + sanity check (NO truncation — the image alone is 256 tokens)

In [ ]:
def build_batch(row):
    image = load_img(row["filename"])
    prompt = [{"role": "user", "content": [{"type": "image", "image": image},
                                           {"type": "text", "text": PROMPT_TEXT}]}]
    prompt_len = processor.apply_chat_template(prompt, add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt")["input_ids"].shape[-1]
    full_msgs = prompt + [{"role": "assistant", "content": [{"type": "text", "text": row["processed_report"]}]}]
    full = processor.apply_chat_template(full_msgs, add_generation_prompt=False, tokenize=True,
                                         return_dict=True, return_tensors="pt")   # no truncation
    keep = ("input_ids", "attention_mask", "pixel_values", "token_type_ids")      # token_type_ids is required by Gemma3
    batch = {k: v.to(model.device) for k, v in full.items() if k in keep}
    batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)
    labels = batch["input_ids"].clone()
    labels[:, :prompt_len] = -100
    return batch, labels

ft_train_df = (train_df[train_df["projection"] == "Frontal"].dropna(subset=["processed_report"])
               .sample(frac=1, random_state=42).reset_index(drop=True))
print("train samples:", len(ft_train_df))

In [ ]:
batch, labels = build_batch(ft_train_df.iloc[0])
print("keys:", list(batch.keys()))
print("seq len:", batch["input_ids"].shape[-1], "| supervised tokens:", (labels != -100).sum().item())
assert "token_type_ids" in batch, "token_type_ids missing"
assert (labels != -100).sum() > 5, "labels are all masked"
model.eval()
with torch.no_grad():
    print("loss:", model(**batch, labels=labels).loss.item())
del batch, labels; torch.cuda.empty_cache()

## 7. Train in chunks — re-run the chunk cell until it says DONE

In [ ]:
import bitsandbytes as bnb

TOTAL_STEPS, GRAD_ACCUM, LR = 400, 4, 1e-4
CHUNK, SAVE_EVERY = 100, 25          # CHUNK must be a multiple of GRAD_ACCUM
assert CHUNK % GRAD_ACCUM == 0

state_path = SAVE_DIR / "train_state.json"
state = json.load(open(state_path)) if state_path.exists() else {"step": 0, "losses": []}
print("resuming at step", state["step"])

optimizer = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=LR)

In [ ]:
def save_progress():
    model.save_pretrained(str(CKPT))
    json.dump(state, open(state_path, "w"))

model.train(); model.config.use_cache = False
start, end = state["step"], min(state["step"] + CHUNK, TOTAL_STEPS)
fails = 0
pbar = tqdm(range(start, end), desc=f"steps {start}-{end}")
for step in pbar:
    try:
        batch, labels = build_batch(ft_train_df.iloc[step % len(ft_train_df)])
        loss = model(**batch, labels=labels).loss
        assert torch.isfinite(loss), "non-finite loss"
        (loss / GRAD_ACCUM).backward()
        state["losses"].append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.3f}")
        del batch, labels, loss
    except Exception as e:
        fails += 1
        print(f"[step {step}] {type(e).__name__}: {e}")
        optimizer.zero_grad(); torch.cuda.empty_cache()
        if fails >= 3:
            raise RuntimeError("3 failures in this chunk — stopping instead of silently skipping") from e
    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step(); optimizer.zero_grad()
    state["step"] = step + 1
    if (step + 1) % SAVE_EVERY == 0:
        save_progress()

save_progress()
print("LoRA B norm:", lora_b_norm(model), "| mean loss (last 20):", np.mean(state["losses"][-20:]))
print("DONE" if state["step"] >= TOTAL_STEPS else f"re-run this cell ({state['step']}/{TOTAL_STEPS})")

In [ ]:
import matplotlib.pyplot as plt
plt.plot(pd.Series(state["losses"]).rolling(10).mean()); plt.xlabel("step"); plt.ylabel("loss (10-step avg)"); plt.show()

## 8. Test the fine-tuned model

In [ ]:
model.eval(); model.config.use_cache = True
row = eval_df.iloc[0]
print("Generated (fine-tuned):\n", generate_report(model, processor, load_img(row["filename"])))
print("\nGround truth:\n", row["processed_report"])

In [ ]:
ft_metrics = evaluate_model(model, processor, eval_df, "lora_finetuned")

In [ ]:
base = json.load(open(SAVE_DIR / "metrics_zero_shot.json"))
ft   = json.load(open(SAVE_DIR / "metrics_lora_finetuned.json"))
display(pd.DataFrame([{"stage": "zero_shot", **base}, {"stage": "lora_finetuned", **ft}]))

## 9. Save the final adapter

In [ ]:
model.save_pretrained(str(SAVE_DIR / "medgemma-cxr-lora"))
processor.save_pretrained(str(SAVE_DIR / "medgemma-cxr-lora"))
print("Saved to", SAVE_DIR / "medgemma-cxr-lora")